# Keithley 2460 M4 local-runtime commissioning

This notebook prepares the first instrument-local finite-hold test. It is restricted to the nominal 100 ohm resistor on the **front terminals in four-wire configuration**. Do not run it with a battery or unknown DUT.

The notebook verifies TSP mode, loads the reviewed runtime into volatile memory, and provides a separately guarded +1 mA immediate-hold proof. It never sends `*LANG TSP`, resets the instrument, clears the event log, or saves a script to nonvolatile memory. Every hardware action is disabled by default.

In [ ]:
from __future__ import annotations

import json
import time
from dataclasses import asdict

from ophyd_electrochemistry.acquisition import StartMode
from ophyd_electrochemistry.keithley.k2460 import (
    CurrentHoldProof,
    Keithley2460Config,
    M4RuntimeController,
    PyVisaKeithley2460Transport,
    RuntimeState,
    SafetyConfig,
    TimingPolicy,
    VisaTransportConfig,
    packaged_runtime,
)
from ophyd_electrochemistry.keithley.k2460.commissioning import run_read_only_diagnostic

RESOURCE = "TCPIP0::169.254.113.151::5025::SOCKET"
RUN_QUERY_ONLY_TSP_PREFLIGHT = False
INSTALL_VOLATILE_RUNTIME = False
RUN_IMMEDIATE_HOLD = False
CONFIRM_FRONT_4WIRE_100_OHM_AND_OUTPUT_OFF = False
CONFIRM_TSP_SELECTED_AND_REBOOTED = False

## Offline artifact inspection

This cell reads only the installed package resource. It does not open a network connection.

In [ ]:
artifact = packaged_runtime()
assert artifact.abi == "oe-k2460-m4-hold-v1"
assert artifact.script_name == f"oe_m4_{artifact.sha256[:24]}"
assert "\n\n" not in artifact.source
assert "trigger.BLOCK_SOURCE_OUTPUT, smu.ON" in artifact.source
assert "trigger.BLOCK_SOURCE_OUTPUT, smu.OFF" in artifact.source
assert "delay(" not in artifact.source
print({"abi": artifact.abi, "build": artifact.build, "sha256": artifact.sha256})

## Query-only TSP preflight

After selecting TSP at the front panel and rebooting, update `RESOURCE` if the link-local address changed and set `RUN_QUERY_ONLY_TSP_PREFLIGHT = True`. This sends only reviewed identity, language, and output-state queries.

In [ ]:
if not RUN_QUERY_ONLY_TSP_PREFLIGHT:
    raise RuntimeError("STOP: query-only hardware preflight is disabled by default.")

tsp_config = VisaTransportConfig(
    resource_name=RESOURCE,
    backend="@py",
    required_language="TSP",
    io_timeout_ms=5_000,
)
preflight = run_read_only_diagnostic(tsp_config)
assert preflight.output_enabled is False
print(json.dumps(preflight.as_dict(), indent=2, sort_keys=True))

## Physical confirmation before installation

If the instrument is still in SCPI, stop and use **MENU → System → Settings → Command Set → TSP**, then reboot. Do not switch language remotely.

Confirm the nominal 100 ohm resistor is still on the front terminals in four-wire configuration and the OUTPUT indicator is OFF. Installation initializes output OFF but does not prepare, arm, or source a hold. Leave the event log intact.

In [ ]:
if not (CONFIRM_FRONT_4WIRE_100_OHM_AND_OUTPUT_OFF and CONFIRM_TSP_SELECTED_AND_REBOOTED):
    raise RuntimeError(
        "STOP: wait for the physical setup and front-panel TSP/reboot confirmations."
    )

if not INSTALL_VOLATILE_RUNTIME:
    raise RuntimeError("STOP: volatile runtime installation is disabled by default.")

config = Keithley2460Config(
    visa_resource=RESOURCE,
    source_terminal="front",
    sense="remote",
    safety=SafetyConfig(
        voltage_min_v=-0.2,
        voltage_max_v=0.2,
        charge_current_max_a=0.001,
        discharge_current_max_a=0.001,
        power_abs_max_w=0.001,
        source_off_mode="normal",
    ),
    timing=TimingPolicy(
        command_timeout_s=10,
        external_start_timeout_s=2,
        shutdown_timeout_s=2,
        poll_period_s=0.01,
        required_abort_latency_s=1,
        required_cutoff_latency_s=1,
    ),
)
transport = PyVisaKeithley2460Transport(
    VisaTransportConfig(
        resource_name=RESOURCE,
        backend="@py",
        required_language="TSP",
        io_timeout_ms=10_000,
        script_line_delay_s=0.01,
    )
)
try:
    transport.connect()
    controller = M4RuntimeController(transport, config)
    installed = controller.install()
    status = controller.status()
    assert status.state.value == "idle" and status.output_enabled is False
    print(
        json.dumps(
            {"runtime": asdict(installed), "status": asdict(status)},
            default=str,
            indent=2,
            sort_keys=True,
        )
    )
finally:
    transport.close()

## Separately guarded immediate hold

Successful installation proves compile/load/initialize and idle status only. Set `RUN_IMMEDIATE_HOLD = True` only after freshly confirming the physical resistor setup and OUTPUT-off state. The proof uses +1 mA, a fixed 1 mA range, 0.2 V limit, NORMAL off mode, and a 250 ms instrument-local delay. It requires PREPARED/output-off before arm, COMPLETE/output-off afterward, and recovery to IDLE/output-off/0 A. Warning and error counts must not increase; normal operation may add informational events.

In [ ]:
if not RUN_IMMEDIATE_HOLD:
    raise RuntimeError("STOP: immediate hold is disabled pending explicit physical confirmation.")
if not CONFIRM_FRONT_4WIRE_100_OHM_AND_OUTPUT_OFF:
    raise RuntimeError("STOP: front-terminal four-wire resistor setup is not confirmed.")
if not CONFIRM_TSP_SELECTED_AND_REBOOTED:
    raise RuntimeError("STOP: TSP selection and reboot are not confirmed.")

transport = PyVisaKeithley2460Transport(
    VisaTransportConfig(
        resource_name=RESOURCE,
        backend="@py",
        required_language="TSP",
        io_timeout_ms=5_000,
    )
)
controller = None
try:
    transport.connect()
    controller = M4RuntimeController(transport, config)
    installed = controller.install()
    initial = controller.status()
    assert initial.state == RuntimeState.IDLE and not initial.output_enabled
    counts_before = {
        "all": int(transport.query("print(eventlog.getcount(eventlog.SEV_ALL))")),
        "warnings": int(transport.query("print(eventlog.getcount(eventlog.SEV_WARN))")),
        "errors": int(transport.query("print(eventlog.getcount(eventlog.SEV_ERROR))")),
    }
    hold = CurrentHoldProof(
        current_a=0.001,
        source_range_a=0.001,
        voltage_limit_v=0.2,
        duration_s=0.25,
        start_mode=StartMode.IMMEDIATE,
    )
    prepared = controller.prepare_current_hold(hold)
    assert prepared.state == RuntimeState.PREPARED and not prepared.output_enabled
    armed = controller.arm()
    time.sleep(0.5)
    completed = controller.status()
    assert completed.state == RuntimeState.COMPLETE and not completed.output_enabled
    recovered = controller.recover()
    final_output = transport.query("print(smu.source.output)")
    final_level = float(transport.query("print(smu.source.level)"))
    counts_after = {
        "all": int(transport.query("print(eventlog.getcount(eventlog.SEV_ALL))")),
        "warnings": int(transport.query("print(eventlog.getcount(eventlog.SEV_WARN))")),
        "errors": int(transport.query("print(eventlog.getcount(eventlog.SEV_ERROR))")),
    }
    assert recovered.state == RuntimeState.IDLE and not recovered.output_enabled
    assert final_output == "smu.OFF" and final_level == 0.0
    assert counts_after["warnings"] == counts_before["warnings"]
    assert counts_after["errors"] == counts_before["errors"]
    print(
        json.dumps(
            {
                "runtime": asdict(installed),
                "hold": asdict(hold),
                "prepared": asdict(prepared),
                "armed": asdict(armed),
                "completed": asdict(completed),
                "recovered": asdict(recovered),
                "event_counts_before": counts_before,
                "event_counts_after": counts_after,
            },
            default=str,
            indent=2,
            sort_keys=True,
        )
    )
except BaseException:
    if controller is not None and transport.connected:
        try:
            controller.force_safe()
        except BaseException:
            pass
    raise
finally:
    transport.close()